# Model Comparison — 6 architectures, each on its OWN legacy recipe

Trains **6 multimodal architectures** (`MM-Simple`, `MM-Augmentation`,
`MM-Attention-Augmentation`, `MM-Transformer`, `MM-Transformer-Attention`,
`MM-ViT16`) for each of 3 seeds (`42, 1, 2`) — **18 trainings** total.

**All six are multimodal** (elastography image + kPa Elastic-Modulus value). Each
architecture was verified against **five legacy notebook versions**, and the
definitive one identified by its recorded `model.summary()` parameter-count
fingerprint. Each model here reproduces that definitive design **and its own
legacy training recipe** — image resolution, optimizer, learning-rate schedule,
epoch budget, early-stopping, and preprocessing (see `MODEL_CFG`):

| Arch | Params | Image | Optimizer | Epochs |
|------|--------|-------|-----------|--------|
| MM-Simple | 165,923 | 300×400 | SGD 1e-3 (m .9, nesterov) | 50 |
| MM-Augmentation | 165,923 | 300×400 | SGD 3.5e-3 + ReduceLROnPlateau | 100 |
| MM-Attention-Augmentation | 15,273,075 | 300×400 | SGD 3.5e-3 (m .9, nesterov) | 100 |
| MM-Transformer | 373,667 | 300×400 | SGD 1e-3 (m .9, nesterov) | 70 |
| MM-Transformer-Attention | 16,059,795 | 300×400 | AdamW 2e-4 + cosine-restarts | 100 |
| MM-ViT16 | 315,011 | 304×400 | SGD 1e-3 (m .9, nesterov) | 100 |

The **only** deliberately-unified knob is data augmentation — set to the
manuscript's stated spec (§27) for every model: horizontal flip, rotation 0.05,
translation 0.05, contrast 0.05. kPa is z-scored (`StandardScaler`, fit on train).

For a fixed seed, every model is evaluated on the **identical test samples**
(same filenames, same order), so the saved per-sample predictions are row-aligned
across models — which is what makes DeLong's test and paired bootstrap valid
downstream, even though the image tensors differ in resolution (300 vs 304).

Everything needed for the statistical comparison (`Model_Comparison_statistics.ipynb`)
is saved under `./model_comparison_results/` and can be downloaded as a zip (final cell).

## 0. Environment (RunPod / GPU)

In [ ]:
# ─── Suppress warnings (show errors only) ───────────────────────────────────
import warnings, os
warnings.filterwarnings("ignore")          # hide all Python/sklearn UserWarnings
os.environ["PYTHONWARNINGS"] = "ignore"
try:
    import tensorflow as _tf
    _tf.get_logger().setLevel("ERROR")     # hide TF INFO/WARNING chatter
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
except Exception:
    pass


In [ ]:
# Uncomment on a fresh pod:
!pip install -q openpyxl scipy scikit-learn comet_ml pillow

## 1. Configuration

In [ ]:
# ─── Paths (EDIT THESE FOR RUNPOD) ──────────────────────────────────────────
IMAGES_DIR = "./Elastography_images"     # folder with response/ stable/ non-response/ subfolders
KPA_EXCEL  = "./clustering_all_v5.xlsx"   # Excel with 'name' + 'Elastic Modulus SWE (kPa)'

# ─── Output tree ────────────────────────────────────────────────────────────
from pathlib import Path
OUTPUT_DIR = Path("./model_comparison_results")
DIR_TABLES   = OUTPUT_DIR / "tables"
DIR_PREDS    = OUTPUT_DIR / "predictions"
DIR_HIST     = OUTPUT_DIR / "histories"
DIR_MODELS   = OUTPUT_DIR / "models"
DIR_SUMM     = OUTPUT_DIR / "summaries"
DIR_REPORTS  = OUTPUT_DIR / "reports"
DIR_CONFMAT  = OUTPUT_DIR / "confusion"
DIR_NORM     = OUTPUT_DIR / "normalizers"
DIR_FIG      = OUTPUT_DIR / "figures"
DIR_CONFIG   = OUTPUT_DIR / "config"
DIR_SPLITS   = DIR_CONFIG / "splits"
for d in (DIR_TABLES, DIR_PREDS, DIR_HIST, DIR_MODELS, DIR_SUMM, DIR_REPORTS,
          DIR_CONFMAT, DIR_NORM, DIR_FIG, DIR_CONFIG, DIR_SPLITS):
    d.mkdir(parents=True, exist_ok=True)
print("Output tree under", OUTPUT_DIR.resolve())

# ─── The 6 architectures to compare ─────────────────────────────────────────
ARCHS = [
    "MM-Simple",
    "MM-Augmentation",
    "MM-Attention-Augmentation",
    "MM-Transformer",
    "MM-Transformer-Attention",
    "MM-ViT16",
]

# ─── Seeds for the shared split + confidence intervals ──────────────────────
# For a given seed ALL 6 architectures see the identical train/val/test SAMPLES
# (same filenames, same split indices), so their saved test predictions are
# row-aligned per sample — valid for DeLong / paired bootstrap — even though each
# architecture resizes the image to its own legacy resolution internally.
SEEDS = [42, 1, 2]

# ─── All six legacy models are MULTIMODAL (image + kPa) ─────────────────────
USES_KPA = {a: True for a in ARCHS}

# ─── Split fractions (stratified by class) ──────────────────────────────────
TEST_FRAC = 0.15
VAL_REL   = 0.15 / (1 - TEST_FRAC)

CLASS_NAMES = ["response", "stable", "non-response"]
NUM_CLASSES = len(CLASS_NAMES)

# Images are LOADED once at the largest resolution any model needs (304, a
# multiple of PATCH_SIZE 16 for ViT16). Each model then resizes to its OWN legacy
# size in the tf.data pipeline (300x400 for the CNN/transformer arms, 304x400 ViT16).
LOAD_HEIGHT, LOAD_WIDTH = 304, 400
PATCH_SIZE = 16

# ─── Transformer hyperparameters (MM-Transformer & MM-ViT16 arms) ───────────
TRANSFORMER_HEADS     = 4
TRANSFORMER_HEAD_SIZE = 32
TRANSFORMER_FF_DIM    = 128
TRANSFORMER_LAYERS    = 2
WEIGHT_DECAY          = 1e-4
DROPOUT_RATE          = 0.1
PROJECTION_DIM        = TRANSFORMER_HEAD_SIZE * TRANSFORMER_HEADS   # 128

BATCH_SIZE = 32

# ─── Data source: COMET artifact (same as the ablation notebook) ────────────
USE_COMET      = True
COMET_PROJECT  = "multi_modal_development"
COMET_ARTIFACT = "elastography_images_merged"

# ─── PER-MODEL FAITHFUL TRAINING RECIPE ─────────────────────────────────────
# Confirmed against 5 legacy notebook versions per architecture — each version's
# rendered model.summary() Total-params fingerprint pinned the definitive one.
# Every model reproduces its OWN legacy image size, optimizer, LR schedule, epoch
# budget, early-stopping, and preprocessing. kPa is StandardScaler-scaled for all.
# The ONLY deliberately-unified knob is AUGMENTATION, set to the manuscript's
# stated spec (§27) for every model:
#   RandomFlip("horizontal"), RandomRotation(0.05), RandomTranslation(0.05,0.05),
#   RandomContrast(0.05).
# Fields:
#   img_h/img_w      legacy input resolution (300 CNN/transformer, 304 ViT16)
#   adjust_contrast  legacy preprocess applied tf.image.adjust_contrast(img,0.5)?
#   optimizer        "sgd" | "adamw"  (resolved to a fresh optimizer per run)
#   lr               initial learning rate
#   momentum,nesterov  SGD only ; weight_decay  AdamW only
#   schedule         "cosine" | "cosine_restarts"
#   epochs           legacy epoch budget
#   es_patience,es_start  EarlyStopping(patience, start_from_epoch)
#   reduce_lr        None, or patience for ReduceLROnPlateau(factor=0.5, min_lr=1e-7)
MODEL_CFG = {
    "MM-Simple": dict(
        img_h=300, img_w=400, adjust_contrast=False,
        optimizer="sgd", lr=1e-3, momentum=0.9, nesterov=True,
        schedule="cosine", epochs=50, es_patience=5, es_start=20, reduce_lr=None,
    ),
    "MM-Augmentation": dict(
        img_h=300, img_w=400, adjust_contrast=True,
        optimizer="sgd", lr=3.5e-3, momentum=0.0, nesterov=False,
        schedule="cosine", epochs=100, es_patience=25, es_start=0, reduce_lr=25,
    ),
    "MM-Attention-Augmentation": dict(
        img_h=300, img_w=400, adjust_contrast=True,
        optimizer="sgd", lr=3.5e-3, momentum=0.9, nesterov=True,
        schedule="cosine", epochs=100, es_patience=20, es_start=20, reduce_lr=None,
    ),
    "MM-Transformer": dict(
        img_h=300, img_w=400, adjust_contrast=True,
        optimizer="sgd", lr=1e-3, momentum=0.9, nesterov=True,
        schedule="cosine", epochs=70, es_patience=20, es_start=20, reduce_lr=None,
    ),
    "MM-Transformer-Attention": dict(
        # Published GitHub recipe: AdamW with NO LR schedule (only ReduceLROnPlateau),
        # EarlyStopping patience 15 (no start_from_epoch). 5-fold in the source; here
        # trained on the shared single split like the others.
        img_h=300, img_w=400, adjust_contrast=True,
        optimizer="adamw", lr=2e-4, weight_decay=1e-4,
        schedule="none", epochs=100, es_patience=15, es_start=0, reduce_lr=8,
    ),
    "MM-ViT16": dict(
        img_h=304, img_w=400, adjust_contrast=False,
        optimizer="sgd", lr=1e-3, momentum=0.9, nesterov=True,
        schedule="cosine", epochs=100, es_patience=20, es_start=20, reduce_lr=None,
    ),
}

KPA_TRANSFORM = "standard"   # StandardScaler (z-score), fit on TRAIN only

print("Architectures:", ARCHS)
print("Seeds:", SEEDS)
print(f"Total trainings: {len(ARCHS) * len(SEEDS)}")
for a in ARCHS:
    c = MODEL_CFG[a]
    print(f"  {a:26s} {c['img_h']}x{c['img_w']}  {c['optimizer']}@{c['lr']:.0e}  "
          f"{c['schedule']}  {c['epochs']}ep")


## 2. Imports & GPU check

In [ ]:
import os, json, joblib
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import (
    layers, regularizers, optimizers, callbacks, Input, Model, backend as K
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.utils import shuffle as sk_shuffle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, log_loss,
    confusion_matrix, classification_report, precision_recall_fscore_support,
)

gpus = tf.config.list_physical_devices("GPU")
DEVICE = "/GPU:0" if gpus else "/CPU:0"
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
print("Using device:", DEVICE)


## 3. Load data once (images + kPa + labels)

Loaded EXACTLY as in the ablation notebook: the COMET artifact provides the
class names (`logged_artifact.metadata["classes"]`) and `flow_from_directory`
loads all images in one batch.

In [ ]:
# ─── Comet experiment + image artifact (EXACT pattern from the ViT16 notebooks)
from comet_ml import Experiment

experiment = Experiment(
    api_key=os.getenv("COMET_API_KEY"),
    project_name=COMET_PROJECT,
    auto_histogram_weight_logging=True,
    auto_histogram_gradient_logging=True,
    auto_histogram_activation_logging=True,
)

# Retrieve the dataset artifact; download to ./ (creates ./Elastography_images)
logged_artifact = experiment.get_artifact("elastography_images_merged")
if not os.path.exists("./Elastography_images"):
    local_artifact = logged_artifact.download("./")

In [ ]:
# ─── Load all images ONCE at the max resolution (304x400) ───────────────────
# Class names come from the artifact metadata (not hardcoded). Each model later
# resizes to its own legacy size inside the tf.data pipeline.
class_names = logged_artifact.metadata["classes"]
assert class_names == CLASS_NAMES, (
    f"Artifact classes {class_names} != configured CLASS_NAMES {CLASS_NAMES}; "
    "align CLASS_NAMES in config to the artifact order.")

dataset = ImageDataGenerator().flow_from_directory(
    "./Elastography_images",
    batch_size=1579,                 # one big batch = load everything
    class_mode="sparse",
    target_size=(LOAD_HEIGHT, LOAD_WIDTH),
    shuffle=False,
    classes=class_names,
)
with tf.device(DEVICE):
    x_all, y_all = dataset.next()

basenames = [os.path.basename(p) for p in dataset.filenames]
print("Classes:", class_names)
print(f"Loaded {x_all.shape[0]} images, shape {x_all.shape}")
assert x_all.shape[0] > 0, "flow_from_directory found 0 images — check the artifact downloaded."


In [ ]:
# ─── Load kPa values and align to image order ───────────────────────────────
df = pd.read_excel(KPA_EXCEL)
df_small = df[["name", "Elastic Modulus SWE (kPa)"]].copy()
df_small.columns = ["filename", "elastic_modulus_kpa"]
# Normalize a Greek capital Tau sometimes present in filenames
df_small["filename"] = df_small["filename"].str.replace("\u03A4", "T", regex=False)

num_map = dict(zip(df_small["filename"], df_small["elastic_modulus_kpa"]))
missing = [b for b in basenames if b not in num_map]
assert not missing, f"{len(missing)} images missing kPa in Excel: {missing[:5]}"

X_num_all = np.array([num_map[b] for b in basenames], dtype=np.float32).reshape(-1, 1)
basenames_arr = np.array(basenames)

# Sanity checks
assert x_all.shape[0] == X_num_all.shape[0] == y_all.shape[0]
print("kPa range:", round(float(X_num_all.min()), 2), "-", round(float(X_num_all.max()), 2))
print("Class counts:", {CLASS_NAMES[i]: int((y_all == i).sum()) for i in range(NUM_CLASSES)})
print("\u2705 Images, kPa, and labels aligned.")

## 4. Helpers: split, kPa normalizer, data pipeline

In [ ]:
# ─── kPa normalization: StandardScaler (z-score), fit on TRAIN only ─────────
# Every legacy model z-scored the kPa value with sklearn StandardScaler fit on
# the training split only, then transformed val/test. We fit ONE scaler per seed
# (the kPa values and split are shared across all architectures for that seed).
def fit_kpa_scaler(Xn_train):
    scaler = StandardScaler()
    scaler.fit(np.asarray(Xn_train, dtype=np.float64))
    return scaler

def apply_kpa_scaler(scaler, Xn):
    return scaler.transform(np.asarray(Xn, dtype=np.float64)).astype(np.float32)


In [ ]:
# ─── ONE shared stratified split, reproducible per seed ─────────────────────
# For a given seed this returns the SAME train/val/test for EVERY architecture,
# so all 6 models are evaluated on the identical test images in identical order.
# That per-sample alignment is what makes DeLong's test valid across models.
def make_split(seed):
    Xi, Xn, Y, idx, fn = sk_shuffle(
        x_all, X_num_all, y_all, np.arange(len(y_all)), basenames_arr,
        random_state=seed,
    )
    (Xi_tv, Xi_te, Xn_tv, Xn_te, Y_tv, Y_te, fn_tv, fn_te) = train_test_split(
        Xi, Xn, Y, fn, test_size=TEST_FRAC, random_state=seed,
        shuffle=True, stratify=Y,
    )
    (Xi_tr, Xi_va, Xn_tr, Xn_va, Y_tr, Y_va, fn_tr, fn_va) = train_test_split(
        Xi_tv, Xn_tv, Y_tv, fn_tv, test_size=VAL_REL, random_state=seed,
        shuffle=True, stratify=Y_tv,
    )
    return dict(
        Xi_tr=Xi_tr, Xn_tr=Xn_tr, Y_tr=Y_tr, fn_tr=fn_tr,
        Xi_va=Xi_va, Xn_va=Xn_va, Y_va=Y_va, fn_va=fn_va,
        Xi_te=Xi_te, Xn_te=Xn_te, Y_te=Y_te, fn_te=fn_te,
    )

In [ ]:
# ─── tf.data pipeline — PER-MODEL preprocessing + shared augmentation ────────
# Images are loaded once at 304x400; each model resizes to ITS legacy resolution
# and (except ViT16) applies the legacy fixed tf.image.adjust_contrast(img, 0.5).
# AUGMENTATION is unified to the manuscript spec (§27) for every model.
AUTOTUNE = tf.data.AUTOTUNE

# Manuscript augmentation (train only): horizontal flip, small rotation/translation,
# modest contrast. No brightness / vertical flip / colour-space jitter (those
# corrupt elastography's quantitative colour-encoded stiffness).
_data_augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.05),
], name="manuscript_augmentation")

def _make_preprocess(img_h, img_w, adjust_contrast):
    def _preprocess(image, numeric, label):
        image = tf.image.resize(image, [img_h, img_w])
        image = tf.cast(image, tf.float32) / 255.0
        if adjust_contrast:
            image = tf.image.adjust_contrast(image, 0.5)   # legacy fixed step
        image = tf.clip_by_value(image, 0.0, 1.0)
        numeric = tf.cast(numeric, tf.float32)
        label = tf.cast(label, tf.int64)
        return (image, numeric), label
    return _preprocess

def _augment(inputs, label):
    image, numeric = inputs
    image = _data_augment(image)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return (image, numeric), label

def make_dataset(images, numerics, labels, seed, img_h, img_w, adjust_contrast,
                 shuffle=False, augment=False, cache=False):
    with tf.device("/CPU:0"):
        ds = tf.data.Dataset.from_tensor_slices((images, numerics, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(images), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(_make_preprocess(img_h, img_w, adjust_contrast), num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(_augment, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


## 5. Model definitions

Every builder returns a `keras.Model` whose inputs are `[image_input, num_input]`
at that model's **own legacy resolution** (`MODEL_CFG[arch]`). All six are
multimodal — each fuses the image with the kPa value. Builders were verified
layer-by-layer against the definitive legacy notebook (confirmed across 5 versions)
and reproduce each recorded parameter count (see §5b):

| Arch | Legacy fn | Params | Image |
|------|-----------|--------|-------|
| MM-Simple | `build_multi_modal` | 165,923 | 300×400 |
| MM-Augmentation | `build_augm_cnn_multi` (= Simple) | 165,923 | 300×400 |
| MM-Attention-Augmentation | `build_multi_modal_attention` | 15,273,075 | 300×400 |
| MM-Transformer | `build_multi_modal_transformer` | 373,667 | 300×400 |
| MM-Transformer-Attention | `build_combined_multimodal_model` | 16,059,795 | 300×400 |
| MM-ViT16 | `build_multi_modal_transformer` (ViT) | 315,011 | 304×400 |

MM-Simple and MM-Augmentation share the same architecture; they differ only in
their legacy training recipe. MM-Transformer keeps its post-transformer LayerNorm
(`post_tx_ln`). The attention arms use `Flatten`, so their param count is tied to
the 300×400 resolution.

In [ ]:
# ─── Custom layers (ViT / transformer arms) ─────────────────────────────────
@tf.keras.utils.register_keras_serializable()
class TokenSlice(layers.Layer):
    """Pick token at `index` from (batch, seq, dim)."""
    def __init__(self, index=0, **kw):
        super().__init__(**kw); self.index = index
    def call(self, x):
        return x[:, self.index, :]
    def get_config(self):
        return {**super().get_config(), "index": self.index}

@tf.keras.utils.register_keras_serializable()
class CLSToken(layers.Layer):
    """Learnable [CLS] token prepended to a patch sequence (image-only ViT)."""
    def __init__(self, dim, **kw):
        super().__init__(**kw); self.dim = dim
    def build(self, input_shape):
        self.cls = self.add_weight(name="cls", shape=(1, 1, self.dim),
                                   initializer="random_normal", trainable=True)
    def call(self, x):
        batch = tf.shape(x)[0]
        return tf.concat([tf.tile(self.cls, [batch, 1, 1]), x], axis=1)
    def get_config(self):
        return {**super().get_config(), "dim": self.dim}

@tf.keras.utils.register_keras_serializable()
class ParametrisedCompatibility(layers.Layer):
    """Attention compatibility score between local features and a global vector.
    Used by the MM-Attention-Augmentation and MM-Transformer-Attention arms."""
    def __init__(self, kernel_regularizer=None, **kwargs):
        super().__init__(**kwargs)
        self.kernel_regularizer = regularizers.get(kernel_regularizer)
    def build(self, input_shape):
        C = input_shape[0][-1]
        self.u = self.add_weight(
            name="u", shape=(C, 1), initializer="uniform",
            regularizer=self.kernel_regularizer, trainable=True,
        )
        super().build(input_shape)
    def call(self, inputs):
        local, g = inputs
        g_proj = K.expand_dims(K.expand_dims(g, 1), 1)
        combined = tf.nn.tanh(local + g_proj)
        return K.dot(combined, self.u)
    def compute_output_shape(self, input_shape):
        return (input_shape[0][0], input_shape[0][1], input_shape[0][2], 1)
    def get_config(self):
        config = super().get_config()
        config.update({"kernel_regularizer": regularizers.serialize(self.kernel_regularizer)})
        return config

CUSTOM_OBJECTS = {
    "TokenSlice": TokenSlice,
    "CLSToken": CLSToken,
    "ParametrisedCompatibility": ParametrisedCompatibility,
}

In [ ]:
# ─── Shared transformer encoder + ViT patch/stack/head (ViT16 arm) ──────────
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout_rate=0.1):
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads,
                                  dropout=dropout_rate)(x, x)
    x = layers.Dropout(dropout_rate)(x)
    res = layers.Add()([inputs, x])
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.Dense(ff_dim, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(inputs.shape[-1])(x)
    return layers.Add()([res, x])


def _patch_tokens(image_input):
    """Image -> (batch, num_patches, PROJECTION_DIM) via a strided patch conv."""
    x = layers.Conv2D(
        filters=PROJECTION_DIM, kernel_size=(PATCH_SIZE, PATCH_SIZE),
        strides=(PATCH_SIZE, PATCH_SIZE), padding="valid",
        kernel_regularizer=regularizers.l2(WEIGHT_DECAY), name="patch_proj",
    )(image_input)
    num_patches = x.shape[1] * x.shape[2]
    return layers.Reshape((num_patches, PROJECTION_DIM), name="patch_tokens")(x), num_patches


def _transformer_stack(tokens, seq_len):
    pos_emb = layers.Embedding(input_dim=seq_len, output_dim=PROJECTION_DIM,
                               name="pos_embedding")
    positions = tf.range(start=0, limit=seq_len, delta=1)[tf.newaxis, :]
    x = layers.Add(name="add_pos")([tokens, pos_emb(positions)])
    for _ in range(TRANSFORMER_LAYERS):
        x = transformer_encoder(x, TRANSFORMER_HEAD_SIZE, TRANSFORMER_HEADS,
                                TRANSFORMER_FF_DIM, DROPOUT_RATE)
    return x


def _vit_head(cls):
    cls = layers.LayerNormalization(epsilon=1e-6, name="cls_norm")(cls)
    h = layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                     name="cls_embedding")(cls)
    h = layers.Dropout(DROPOUT_RATE, name="cls_dropout")(h)
    return layers.Dense(NUM_CLASSES, activation="softmax",
                        kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                        name="predictions")(h)

In [ ]:
# ─── CNN backbones (MM-Simple, MM-Augmentation) ─────────────────────────────
def _simple_cnn_tower(img_in, weight_decay=1e-4, dropout_rate=0.3):
    """32-32 / 64-64 / 128 conv tower -> GlobalAveragePooling2D vector.
    Matches the legacy `build_cnn_feature_extractor` (MM-Simple / MM-Augmentation).
    GAP makes the parameter count independent of the input resolution."""
    x = layers.Conv2D(32, 3, padding="same", kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(weight_decay))(img_in)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, 3, padding="same", kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(x); x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv2D(64, 3, padding="same", kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, 3, padding="same", kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(x); x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv2D(128, 3, padding="same", kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(x); x = layers.Dropout(dropout_rate)(x)

    return layers.GlobalAveragePooling2D(name="global_avg_pool")(x)


def _cnn_multimodal_head(img_feat, num_input, weight_decay=1e-4, dropout_rate=0.3):
    """kPa branch + fusion + head shared by MM-Simple and MM-Augmentation. Legacy
    `build_multi_modal` / `build_augm_cnn_multi`:
      kPa -> Dense(64,relu) -> Dropout ; concat[img(128), num(64)] ->
      Dense(128, l2) -> LeakyReLU(0.1) -> Dropout -> Dense(3, softmax, l2)."""
    y = layers.Dense(64, activation="relu", name="num_dense")(num_input)
    y = layers.Dropout(dropout_rate, name="num_dropout")(y)
    combined = layers.Concatenate(name="fusion")([img_feat, y])
    z = layers.Dense(128, kernel_regularizer=regularizers.l2(weight_decay))(combined)
    z = layers.LeakyReLU(0.1)(z)
    z = layers.Dropout(dropout_rate)(z)
    return layers.Dense(NUM_CLASSES, activation="softmax",
                        kernel_regularizer=regularizers.l2(weight_decay),
                        name="predictions")(z)


def build_mm_simple(image_input, num_input, weight_decay=1e-4, dropout_rate=0.3):
    """Multimodal CNN (USES kPa). Legacy `build_multi_modal` (simple_CNN_multi),
    165,923 params at 300x400."""
    img_feat = _simple_cnn_tower(image_input, weight_decay, dropout_rate)
    return _cnn_multimodal_head(img_feat, num_input, weight_decay, dropout_rate)


def build_mm_augmentation(image_input, num_input, weight_decay=1e-4, dropout_rate=0.3):
    """Multimodal CNN + augmentation (USES kPa). Legacy `build_augm_cnn_multi`
    (augmentation_CNN_multi), 165,923 params at 300x400 — architecturally IDENTICAL
    to MM-Simple; it differs only in its (heavier) legacy training recipe
    (SGD lr 3.5e-3, ReduceLROnPlateau) and the always-on data augmentation."""
    img_feat = _simple_cnn_tower(image_input, weight_decay, dropout_rate)
    return _cnn_multimodal_head(img_feat, num_input, weight_decay, dropout_rate)


In [ ]:
# ─── Attention CNN backbone (MM-Attention-Augmentation) ─────────────────────
def _attention_cnn_features(img_in, reg=0.0005, dropout=0.35):
    """Multi-scale CNN with a global feature vector `g`, returns the three
    attention-pooled vectors + g. Ported from the legacy
    `build_attention_cnn_extractor` / `build_multi_modal_attention`."""
    x = layers.Conv2D(16, 7, padding="same", kernel_initializer="he_normal")(img_in)
    x = layers.Conv2D(16, 5, padding="same", kernel_initializer="he_normal")(x)
    local12 = layers.Activation("relu")(x)

    x = layers.Conv2D(16, 3, padding="same", kernel_initializer="he_normal")(local12)
    local1 = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(local1); x = layers.Dropout(dropout)(x)

    x = layers.Conv2D(32, 3, padding="same", kernel_initializer="he_normal")(x)
    local2 = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(local2); x = layers.Dropout(dropout)(x)

    x = layers.Conv2D(64, 3, padding="same", kernel_initializer="he_normal")(x)
    local3 = layers.Activation("relu")(x)
    x = layers.Conv2D(128, 3, padding="same", kernel_initializer="he_normal")(local3)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(x); x = layers.Dropout(dropout)(x)

    flat = layers.Flatten()(x)
    g = layers.Dense(64, activation="relu")(flat)
    g = layers.Dropout(0.2)(g)
    g = layers.Dense(16, activation="relu", name="global_feat")(g)

    def attention_branch(local_feat, branch_name):
        l = layers.Dense(16)(local_feat)
        c = ParametrisedCompatibility(
            kernel_regularizer=regularizers.l2(reg),
            name=f"cpc_{branch_name}",
        )([l, g])
        flat_c = layers.Flatten(name=f"flatc_{branch_name}")(c)
        a = layers.Activation("softmax", name=f"attn_{branch_name}")(flat_c)
        reshaped = layers.Reshape((-1, 16), name=f"reshape_{branch_name}")(l)
        return layers.Lambda(
            lambda z: K.squeeze(K.batch_dot(K.expand_dims(z[0], 1), z[1]), 1),
            name=f"g_{branch_name}",
        )([a, reshaped])

    g1 = attention_branch(local1, "l1")
    g2 = attention_branch(local2, "l2")
    g12 = attention_branch(local12, "l12")
    return g1, g2, g12, g


def build_mm_attention_augmentation(image_input, num_input, reg=0.0005, dropout=0.35):
    """Multimodal attention CNN (USES kPa). Legacy `build_multi_modal_attention`
    (att_augm_CNN_multi), 15,273,075 params. The three attention-pooled vectors
    are fused WITH a Dense(64) kPa branch (legacy fusion = [g12, g1, g2, y]); the
    global vector `g` is used only inside the compatibility scores, not fused.
    Augmentation is supplied by the shared tf.data pipeline (swe_conservative)."""
    g1, g2, g12, g = _attention_cnn_features(image_input, reg=reg, dropout=dropout)
    # kPa numeric branch (legacy: Dense(64, relu) -> Dropout(0.2))
    y = layers.Dense(64, activation="relu", name="num_dense")(num_input)
    y = layers.Dropout(0.2, name="num_dropout")(y)
    fused = layers.Concatenate(name="fusion")([g12, g1, g2, y])
    z = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(reg))(fused)
    z = layers.Dropout(0.5)(z)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(z)
    return out

In [ ]:
# ─── MM-Transformer (CNN -> 2-token transformer, USES kPa) ──────────────────
def build_mm_transformer(image_input, num_input):
    """CNN 32-32-64-64-128 -> GlobalAvgPool (128) -> [kPa token, image token] ->
    2-token transformer -> post-LayerNorm -> Flatten -> Dense head. (USES kPa.)
    Legacy `build_multi_modal_transformer` (transformer_augm_CNN_multi), 373,667
    params. Confirmed across 4 identical legacy versions:
      - the GAP image vector (already 128) is reshaped DIRECTLY into the image
        token (NO extra image-projection Dense);
      - the kPa scalar IS projected (num_proj Dense 1->128);
      - token order is [num_token, img_token];
      - a post-transformer LayerNormalization (post_tx_ln) precedes Flatten."""
    img_vec = _simple_cnn_tower(image_input, WEIGHT_DECAY, DROPOUT_RATE)  # (B, 128)

    num_proj = layers.Dense(PROJECTION_DIM, activation="relu",
                            kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                            name="num_proj")(num_input)
    num_token = layers.Reshape((1, PROJECTION_DIM), name="num_token")(num_proj)
    img_token = layers.Reshape((1, PROJECTION_DIM), name="img_token")(img_vec)

    seq = layers.Concatenate(axis=1, name="token_concat")([num_token, img_token])  # (B, 2, 128)

    seq_len = 2
    pos_emb = layers.Embedding(input_dim=seq_len, output_dim=PROJECTION_DIM,
                               name="pos_embedding")
    positions = tf.range(start=0, limit=seq_len, delta=1)[tf.newaxis, :]
    x = layers.Add(name="add_pos")([seq, pos_emb(positions)])
    for _ in range(TRANSFORMER_LAYERS):
        x = transformer_encoder(x, TRANSFORMER_HEAD_SIZE, TRANSFORMER_HEADS,
                                TRANSFORMER_FF_DIM, DROPOUT_RATE)

    x = layers.LayerNormalization(epsilon=1e-6, name="post_tx_ln")(x)   # legacy: present
    x = layers.Flatten(name="flatten_tokens")(x)          # (B, 2*128)
    h = layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                     name="head_dense")(x)
    h = layers.Dropout(DROPOUT_RATE)(h)
    out = layers.Dense(NUM_CLASSES, activation="softmax",
                       kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                       name="predictions")(h)
    return out


# ─── MM-Transformer-Attention (attention CNN + patch transformer, USES kPa) ─
def transformer_encoder_block(inputs, head_size, num_heads, ff_dim,
                              dropout_rate=0.1, name_prefix="transformer"):
    x = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_norm1")(inputs)
    attn_output = layers.MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout_rate,
        name=f"{name_prefix}_mha")(x, x)
    attn_output = layers.Dropout(dropout_rate, name=f"{name_prefix}_dropout1")(attn_output)
    x = layers.Add(name=f"{name_prefix}_add1")([inputs, attn_output])
    normed = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_norm2")(x)
    ff_output = layers.Dense(ff_dim, activation="gelu", name=f"{name_prefix}_ff1")(normed)
    ff_output = layers.Dropout(dropout_rate, name=f"{name_prefix}_dropout2")(ff_output)
    ff_output = layers.Dense(inputs.shape[-1], name=f"{name_prefix}_ff2")(ff_output)
    ff_output = layers.Dropout(dropout_rate, name=f"{name_prefix}_dropout3")(ff_output)
    return layers.Add(name=f"{name_prefix}_add2")([x, ff_output])


def build_mm_transformer_attention(image_input, num_input,
                                   patch_size=25, embed_dim=128,
                                   num_transformer_layers=4, num_heads=8,
                                   ff_dim=256, cnn_reg=0.0005, cnn_dropout=0.35,
                                   transformer_dropout=0.15, fusion_dropout=0.3,
                                   final_reg=0.001):
    """CNN + attention + patch transformer, fused with kPa. Legacy
    `build_combined_multimodal_model` (transformer_attention_CNN_multi),
    16,059,795 params at 300x400. Patch 25, embed 128, 4 layers, 8 heads, ff 256,
    CLS token; fusion = [g1,g2,g12,g_global(64)] + transformer(16) + kPa(16)."""
    h_img = image_input.shape[1]
    w_img = image_input.shape[2]

    g1, g2, g12, g_global = _attention_cnn_features(image_input, reg=cnn_reg,
                                                     dropout=cnn_dropout)
    cnn_combined = layers.Concatenate(name="cnn_attention_fusion")([g1, g2, g12, g_global])

    patches = tf.image.extract_patches(
        images=image_input, sizes=[1, patch_size, patch_size, 1],
        strides=[1, patch_size, patch_size, 1], rates=[1, 1, 1, 1], padding="VALID")
    patch_dim = patch_size * patch_size * 3
    num_patches = (h_img // patch_size) * (w_img // patch_size)
    batch_size = tf.shape(image_input)[0]
    patches = tf.reshape(patches, [batch_size, num_patches, patch_dim])

    patch_embeddings = layers.Dense(embed_dim, name="patch_projection")(patches)
    positions = tf.range(start=0, limit=num_patches, delta=1)
    pos_embedding = layers.Embedding(input_dim=num_patches, output_dim=embed_dim,
                                     name="pos_embedding")
    pos_encoding = tf.expand_dims(pos_embedding(positions), axis=0)
    xt = patch_embeddings + pos_encoding

    cls_embedding = layers.Embedding(input_dim=1, output_dim=embed_dim,
                                     name="cls_token_embedding")
    cls_input = tf.zeros((batch_size, 1), dtype=tf.int32)
    cls_tokens = cls_embedding(cls_input)
    xt = layers.Concatenate(axis=1, name="add_cls_token")([cls_tokens, xt])

    for i in range(num_transformer_layers):
        xt = transformer_encoder_block(
            xt, head_size=embed_dim // num_heads, num_heads=num_heads,
            ff_dim=ff_dim, dropout_rate=transformer_dropout,
            name_prefix=f"transformer_layer_{i}")
    cls_representation = xt[:, 0, :]
    transformer_features = layers.Dense(16, activation="relu",
                                        name="transformer_global_features")(cls_representation)

    numeric_processed = layers.Dense(64, activation="relu", name="numeric_dense1")(num_input)
    numeric_processed = layers.Dropout(0.2, name="numeric_dropout1")(numeric_processed)
    numeric_processed = layers.Dense(16, activation="relu", name="numeric_dense2")(numeric_processed)

    fused = layers.Concatenate(name="multimodal_fusion")([
        cnn_combined, transformer_features, numeric_processed])
    x = layers.Dense(128, activation="gelu",
                     kernel_regularizer=regularizers.l2(final_reg),
                     name="classification_dense1")(fused)
    x = layers.Dropout(fusion_dropout, name="classification_dropout1")(x)
    x = layers.Dense(64, activation="gelu",
                     kernel_regularizer=regularizers.l2(final_reg),
                     name="classification_dense2")(x)
    x = layers.Dropout(fusion_dropout, name="classification_dropout2")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax",
                       kernel_regularizer=regularizers.l2(final_reg),
                       name="predictions")(x)
    return out


In [ ]:
# ─── MM-ViT16 (patch-based ViT, USES kPa) + build_model dispatcher ──────────
def build_mm_vit16(image_input, num_input):
    """Patch-based ViT (patch 16). kPa -> token prepended at index 0 to the patch
    tokens, fused by self-attention. Legacy `build_multi_modal_transformer` (ViT),
    315,011 params at 304x400 (19x25=475 patches)."""
    patch_tokens, num_patches = _patch_tokens(image_input)
    num_proj = layers.Dense(PROJECTION_DIM, activation="relu",
                            kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                            name="num_proj")(num_input)
    num_token = layers.Reshape((1, PROJECTION_DIM), name="num_token")(num_proj)
    seq = layers.Concatenate(axis=1, name="token_concat")([num_token, patch_tokens])
    x = _transformer_stack(seq, seq_len=1 + num_patches)
    cls = TokenSlice(index=0, name="cls_pick")(x)
    out = _vit_head(cls)
    return out


def build_model(arch_name):
    """Build the Keras model for a given architecture at ITS legacy input size.

    ALL architectures accept the SAME [image_input, num_input] signature so the
    shared tf.data pipeline works. The image resolution is per-model (MODEL_CFG):
    300x400 for the CNN/transformer arms, 304x400 for MM-ViT16."""
    cfg = MODEL_CFG[arch_name]
    image_input = Input(shape=(cfg["img_h"], cfg["img_w"], 3), name="image_input")
    num_input   = Input(shape=(1,), name="num_input")

    if arch_name == "MM-Simple":
        out = build_mm_simple(image_input, num_input)
    elif arch_name == "MM-Augmentation":
        out = build_mm_augmentation(image_input, num_input)
    elif arch_name == "MM-Attention-Augmentation":
        out = build_mm_attention_augmentation(image_input, num_input)
    elif arch_name == "MM-Transformer":
        out = build_mm_transformer(image_input, num_input)
    elif arch_name == "MM-Transformer-Attention":
        out = build_mm_transformer_attention(image_input, num_input)
    elif arch_name == "MM-ViT16":
        out = build_mm_vit16(image_input, num_input)
    else:
        raise ValueError(f"Unknown architecture: {arch_name}")

    return Model([image_input, num_input], out, name=arch_name.replace("-", "_"))


## 5b. Architecture sanity check — parameter counts vs legacy

Builds each architecture once (no training), saves `model.summary()` to
`summaries/summary_{arch}.txt`, writes `summaries/param_counts.csv`, and compares
the live parameter count to the value recorded for the legacy model in
`04_artifacts/model_statistics_all.xlsx`. A close match confirms each builder is
a faithful port. (The Flatten-based attention arms are expected to be slightly
larger than their legacy figure because they are built at 304×400 rather than the
legacy 300×400 — this is stated in the config cell and is intentional.)

In [ ]:
# ─── Build each architecture once; save summaries + compare param counts ─────
# Recorded legacy parameter counts (04_artifacts/model_statistics_all.xlsx),
# each at that model's OWN legacy image size (300x400 except MM-ViT16 at 304x400).
# With the per-model faithful build these should match EXACTLY.
LEGACY_PARAMS = {
    "MM-Simple":                 165_923,
    "MM-Augmentation":           165_923,
    "MM-Attention-Augmentation": 15_273_075,
    "MM-Transformer":            373_667,
    "MM-Transformer-Attention":  16_059_795,
    "MM-ViT16":                  315_011,
}

_param_rows = []
for _arch in ARCHS:
    keras.backend.clear_session()
    _model = build_model(_arch)
    _total     = int(_model.count_params())
    _trainable = int(sum(np.prod(w.shape) for w in _model.trainable_weights))

    _lines = []
    _model.summary(print_fn=lambda s: _lines.append(s), line_length=110)
    (DIR_SUMM / f"summary_{_arch}.txt").write_text("\n".join(_lines))

    _legacy = LEGACY_PARAMS[_arch]
    _cfg = MODEL_CFG[_arch]
    _param_rows.append({
        "arch": _arch,
        "img": f"{_cfg['img_h']}x{_cfg['img_w']}",
        "total_params": _total,
        "trainable_params": _trainable,
        "legacy_params": _legacy,
        "delta_vs_legacy": _total - _legacy,
        "match": (_total == _legacy),
    })

param_counts = pd.DataFrame(_param_rows)
param_counts.to_csv(DIR_SUMM / "param_counts.csv", index=False)
keras.backend.clear_session()
print("Saved model summaries to", DIR_SUMM)
print("All params match legacy fingerprints:", bool(param_counts["match"].all()))
param_counts


## 6. Train + evaluate one (arch, seed)

In [ ]:
def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    import random as _random
    _random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def compute_metrics(y_true, y_prob):
    """Robust, imbalance-aware metrics; any metric that fails on a degenerate
    split -> NaN so one bad seed never kills the grid."""
    y_pred = y_prob.argmax(axis=1)
    y_bin  = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    out = {}
    def safe(f):
        try:    return f()
        except Exception: return float("nan")
    out["accuracy"]          = safe(lambda: accuracy_score(y_true, y_pred))
    out["balanced_accuracy"] = safe(lambda: balanced_accuracy_score(y_true, y_pred))
    out["macro_f1"]          = safe(lambda: f1_score(y_true, y_pred, average="macro"))
    out["weighted_f1"]       = safe(lambda: f1_score(y_true, y_pred, average="weighted"))
    out["roc_auc_ovr"]       = safe(lambda: roc_auc_score(y_bin, y_prob, average="macro", multi_class="ovr"))
    out["roc_auc_weighted"]  = safe(lambda: roc_auc_score(y_bin, y_prob, average="weighted", multi_class="ovr"))
    out["log_loss"]          = safe(lambda: log_loss(y_true, y_prob, labels=list(range(NUM_CLASSES))))
    for c in range(NUM_CLASSES):
        out[f"auc_{CLASS_NAMES[c]}"] = safe(lambda: roc_auc_score(y_bin[:, c], y_prob[:, c]))
    return out


def build_optimizer_and_callbacks(cfg):
    """Per-model optimizer + LR schedule + callbacks, exactly as in each legacy
    notebook. Returns (optimizer, callbacks_list)."""
    epochs = cfg["epochs"]
    lr     = cfg["lr"]

    if cfg["schedule"] == "cosine_restarts":
        sched = optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=lr, first_decay_steps=30,
            t_mul=1.0, m_mul=0.8, alpha=0.01)
    elif cfg["schedule"] == "cosine":
        sched = optimizers.schedules.CosineDecay(initial_learning_rate=lr, decay_steps=epochs)
    else:  # "none" -> constant LR (rely on ReduceLROnPlateau)
        sched = None

    if cfg["optimizer"] == "adamw":
        opt = optimizers.AdamW(learning_rate=lr, weight_decay=cfg.get("weight_decay", 1e-4),
                               beta_1=0.9, beta_2=0.999, epsilon=1e-8)
    else:  # sgd
        opt = optimizers.SGD(learning_rate=lr, momentum=cfg.get("momentum", 0.0),
                             nesterov=cfg.get("nesterov", False))

    cbs = []
    if sched is not None:
        cbs.append(callbacks.LearningRateScheduler(lambda e, _lr, _s=sched: float(_s(e))))
    cbs.append(callbacks.EarlyStopping(
        monitor="val_loss", patience=cfg["es_patience"],
        restore_best_weights=True, start_from_epoch=cfg["es_start"]))
    if cfg["reduce_lr"] is not None:
        cbs.append(callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=cfg["reduce_lr"], min_lr=1e-7))
    return opt, cbs


PER_CLASS_ROWS = []


def run_arch_seed(arch, seed):
    """Build, compile and train `arch` on the shared split for `seed` using the
    model's OWN legacy recipe (image size, optimizer, schedule, epochs), predict
    on the test set, compute metrics, and save everything downstream needs."""
    cfg = MODEL_CFG[arch]
    set_all_seeds(seed)
    sp = make_split(seed)

    # kPa StandardScaler — fit on this seed's TRAIN only, reused across all archs.
    scaler = fit_kpa_scaler(sp["Xn_tr"])
    Xn_tr = apply_kpa_scaler(scaler, sp["Xn_tr"])
    Xn_va = apply_kpa_scaler(scaler, sp["Xn_va"])
    Xn_te = apply_kpa_scaler(scaler, sp["Xn_te"])

    dh, dw, ac = cfg["img_h"], cfg["img_w"], cfg["adjust_contrast"]
    train_set = make_dataset(sp["Xi_tr"], Xn_tr, sp["Y_tr"], seed, dh, dw, ac,
                             shuffle=True, augment=True)
    val_set   = make_dataset(sp["Xi_va"], Xn_va, sp["Y_va"], seed, dh, dw, ac, cache=True)
    test_set  = make_dataset(sp["Xi_te"], Xn_te, sp["Y_te"], seed, dh, dw, ac, cache=True)

    keras.backend.clear_session()
    set_all_seeds(seed)
    model = build_model(arch)
    total_params     = int(model.count_params())
    trainable_params = int(sum(np.prod(w.shape) for w in model.trainable_weights))

    opt, cbs = build_optimizer_and_callbacks(cfg)
    model.compile(optimizer=opt,
                  loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy"])
    with tf.device(DEVICE):
        history = model.fit(train_set, validation_data=val_set, epochs=cfg["epochs"],
                            callbacks=cbs, verbose=2)

    y_prob = model.predict(test_set, verbose=0)
    # Renormalize rows to sum to 1 (guards against float drift so log_loss /
    # AUC never warn "y_pred values do not sum to one").
    y_prob = np.clip(y_prob.astype(np.float64), 1e-12, 1.0)
    y_prob = y_prob / y_prob.sum(axis=1, keepdims=True)
    y_true = sp["Y_te"].astype(int)
    y_pred = y_prob.argmax(axis=1)
    m = compute_metrics(y_true, y_prob)
    m.update({"arch": arch, "seed": seed,
              "img_h": dh, "img_w": dw, "optimizer": cfg["optimizer"],
              "lr": cfg["lr"], "epochs_cfg": cfg["epochs"],
              "total_params": total_params, "trainable_params": trainable_params,
              "n_test": int(len(y_true)), "n_train": int(len(sp["Y_tr"])),
              "epochs_trained": int(len(history.history["loss"]))})
    print(f"  [{arch} | seed {seed}] acc={m['accuracy']:.3f} "
          f"bal_acc={m['balanced_accuracy']:.3f} macroF1={m['macro_f1']:.3f} "
          f"auc={m['roc_auc_ovr']:.3f} params={total_params:,}")

    tag = f"{arch}_seed{seed}"

    # 1) Aligned per-sample test predictions (row-aligned across archs per seed).
    preds = pd.DataFrame({
        "filename": sp["fn_te"],
        "y_true": y_true,
        "y_pred": y_pred,
        **{f"p_{CLASS_NAMES[c]}": y_prob[:, c] for c in range(NUM_CLASSES)},
    })
    preds.to_csv(DIR_PREDS / f"preds_{tag}.csv", index=False)

    # 2) Training history.
    hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    with open(DIR_HIST / f"history_{tag}.json", "w") as f:
        json.dump(hist, f)

    # 3) Trained model (full + weights).
    model.save(DIR_MODELS / f"model_{tag}.keras")
    model.save_weights(DIR_MODELS / f"model_{tag}.weights.h5")

    # 4) Per-class classification report.
    rep = classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)),
                                target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    pd.DataFrame(rep).transpose().to_csv(DIR_REPORTS / f"classification_report_{tag}.csv")
    prec, rec, f1c, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    for c in range(NUM_CLASSES):
        PER_CLASS_ROWS.append({"arch": arch, "seed": seed, "class": CLASS_NAMES[c],
                               "precision": prec[c], "recall": rec[c], "f1": f1c[c],
                               "support": int(sup[c]), "auc": m[f"auc_{CLASS_NAMES[c]}"]})

    # 5) Confusion matrix (raw counts).
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    pd.DataFrame(cm, index=[f"true_{c}" for c in CLASS_NAMES],
                 columns=[f"pred_{c}" for c in CLASS_NAMES]).to_csv(
        DIR_CONFMAT / f"confusion_{tag}.csv")

    # 6) Fitted kPa scaler.
    joblib.dump(scaler, DIR_NORM / f"kpa_scaler_{tag}.joblib")

    # 7) Split manifest (once per seed).
    split_path = DIR_SPLITS / f"split_seed{seed}.json"
    if not split_path.exists():
        with open(split_path, "w") as f:
            json.dump({"seed": seed,
                       "train": list(map(str, sp["fn_tr"])),
                       "val":   list(map(str, sp["fn_va"])),
                       "test":  list(map(str, sp["fn_te"]))}, f)

    return m


## 7. Run the full grid (architectures × seeds)

Checkpoints after every run to `tables/comparison_runs.csv` and resumes by
skipping `(arch, seed)` pairs already present in that CSV.

In [ ]:
COMPARISON_CSV = DIR_TABLES / "comparison_runs.csv"

# Resume support: skip (arch, seed) pairs already in the CSV.
done = set()
if COMPARISON_CSV.exists():
    prev = pd.read_csv(COMPARISON_CSV)
    done = set(zip(prev["arch"], prev["seed"]))
    print(f"Resuming — {len(done)} runs already complete.")
    records = prev.to_dict("records")
else:
    records = []

for arch in ARCHS:
    for seed in SEEDS:
        if (arch, seed) in done:
            print(f"skip {arch} seed {seed} (done)")
            continue
        print(f"=== TRAIN  arch={arch}  seed={seed} ===")
        rec = run_arch_seed(arch, seed)
        records.append(rec)
        pd.DataFrame(records).to_csv(COMPARISON_CSV, index=False)  # checkpoint

results_df = pd.DataFrame(records)

# Long-form per-class metrics (precision/recall/F1/support/AUC) across all runs.
if PER_CLASS_ROWS:
    pd.DataFrame(PER_CLASS_ROWS).to_csv(DIR_TABLES / "per_class_metrics.csv", index=False)

print("\nAll runs complete.")
results_df

## 8. Save run configuration snapshot

In [ ]:
import sys, subprocess, sklearn, scipy

def _ver(mod, default="unknown"):
    """Version string that works under both Keras 2 and Keras 3 (which drops
    keras.__version__ from the tf.keras shim)."""
    try:
        return mod.__version__
    except Exception:
        try:
            import keras as _k
            return _k.__version__
        except Exception:
            return default

# 1) Run-configuration snapshot (per-model recipe included).
run_config = {
    "archs": ARCHS, "seeds": SEEDS, "uses_kpa": USES_KPA,
    "model_cfg": MODEL_CFG,
    "load_height": LOAD_HEIGHT, "load_width": LOAD_WIDTH, "patch_size": PATCH_SIZE,
    "class_names": CLASS_NAMES,
    "transformer": {"heads": TRANSFORMER_HEADS, "head_size": TRANSFORMER_HEAD_SIZE,
                    "ff_dim": TRANSFORMER_FF_DIM, "layers": TRANSFORMER_LAYERS,
                    "weight_decay": WEIGHT_DECAY, "dropout": DROPOUT_RATE},
    "batch_size": BATCH_SIZE, "kpa_transform": KPA_TRANSFORM,
    "augmentation": "manuscript §27: RandomFlip(h), RandomRotation(0.05), "
                    "RandomTranslation(0.05,0.05), RandomContrast(0.05)",
    "test_frac": TEST_FRAC, "val_rel": VAL_REL,
    "tf_version": _ver(tf), "keras_version": _ver(keras),
    "sklearn_version": sklearn.__version__, "scipy_version": scipy.__version__,
    "numpy_version": np.__version__, "pandas_version": pd.__version__,
    "python_version": sys.version.split()[0],
}
with open(DIR_CONFIG / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)
print("Saved run_config.json")

# 2) Full environment snapshot (pip freeze).
try:
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                            capture_output=True, text=True, timeout=120).stdout
    (DIR_CONFIG / "environment.txt").write_text(freeze)
    print(f"Saved environment.txt ({freeze.count(chr(10))} packages)")
except Exception as e:
    print("Could not capture pip freeze:", e)

# 3) README describing the results tree.
readme = f"""# Model comparison results

Six multimodal architectures (image + kPa) trained on ONE shared stratified split
per seed {SEEDS}. Each model uses its OWN legacy recipe (image size, optimizer, LR
schedule, epochs, preprocessing) — verified against 5 legacy notebook versions per
architecture and reproducing each recorded parameter count. Augmentation is unified
to the manuscript spec (horizontal flip, rotation 0.05, translation 0.05, contrast
0.05). kPa is z-scored with StandardScaler fit on train only.

For a fixed seed, every model is evaluated on the identical test SAMPLES (same
filenames/order), so the per-sample prediction CSVs are row-aligned across models —
which is what makes DeLong's test and paired bootstrap valid, even though the image
tensors differ in resolution (300x400 for CNN/transformer arms, 304x400 for ViT16).

## Folder tree
- predictions/preds_{{arch}}_seed{{seed}}.csv  — per-sample y_true/y_pred/probs (aligned)
- tables/comparison_runs.csv                  — per (arch,seed) metrics + params + img size
- tables/per_class_metrics.csv                — long-form per-class P/R/F1/support/AUC
- reports/classification_report_{{arch}}_seed{{seed}}.csv
- confusion/confusion_{{arch}}_seed{{seed}}.csv
- histories/history_{{arch}}_seed{{seed}}.json
- models/model_{{arch}}_seed{{seed}}.keras (+ .weights.h5)
- summaries/summary_{{arch}}.txt + summaries/param_counts.csv (live vs legacy fingerprint)
- normalizers/kpa_scaler_{{arch}}_seed{{seed}}.joblib
- config/run_config.json (incl. per-model MODEL_CFG), config/environment.txt, config/splits/
- figures/ — populated by the downstream statistics notebook

## Downstream analysis
Run Model_Comparison_statistics.ipynb over predictions/ for the metric inventory,
per-metric plots, ROC + AUC, DeLong p-values, bootstrap 95% CIs, confusion matrices.
"""
(OUTPUT_DIR / "README.md").write_text(readme)
print("Saved README.md")
run_config


## 9. Downstream statistical comparison (separate notebook)

Everything needed for the paired comparison is now on disk under
`./model_comparison_results/`:

- `predictions/preds_{arch}_seed{seed}.csv` — per-sample `y_true`, `y_pred`,
  and class probabilities. **For a fixed seed these are row-aligned across all 6
  architectures** (same `filename` order), so you can load any two models'
  probability columns and run DeLong's test on their one-vs-rest AUCs, or
  bootstrap paired differences in accuracy / macro-F1 directly.
- `tables/comparison_runs.csv` — one row per `(arch, seed)` with all metrics.
- `histories/`, `models/`, `config/splits/` — curves, weights, split manifests.

A downstream analysis notebook can read the prediction CSVs, verify alignment
via the `filename` column, and compute DeLong p-values and bootstrap CIs without
retraining anything.

## 10. Download all results as zip

Bundles the results tree into zip archives for download off the pod. Two zips are
produced so you don't have to move gigabytes of model weights just to get the
tables/predictions:

- **`model_comparison_results.zip`** — everything EXCEPT the `models/` folder
  (predictions, tables, reports, confusion matrices, histories, summaries,
  normalizers, config, README). This is all the statistics notebook needs. Small.
- **`model_comparison_models.zip`** — the trained `models/` only (large; optional).

On Colab these auto-download; on RunPod/Jupyter the cell prints the absolute paths
and sizes plus the `runpodctl` / `scp` command to pull them.

In [ ]:
# ─── Bundle results into zip archives ───────────────────────────────────────
import zipfile

def _zip_dir(zip_path, root, include=None, exclude=None):
    """Zip `root` recursively. `include`/`exclude` are sets of top-level subdir
    names (relative to root) to keep / skip. Returns the zip Path."""
    root = Path(root)
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(root.rglob("*")):
            if not p.is_file():
                continue
            rel = p.relative_to(root)
            top = rel.parts[0] if len(rel.parts) > 1 else rel.parts[0]
            if include is not None and top not in include:
                continue
            if exclude is not None and top in exclude:
                continue
            zf.write(p, arcname=str(Path(root.name) / rel))
    return zip_path

def _human(nbytes):
    for unit in ("B", "KB", "MB", "GB"):
        if nbytes < 1024 or unit == "GB":
            return f"{nbytes:.1f} {unit}"
        nbytes /= 1024

# 1) Everything except the (large) full models — all the stats notebook needs.
results_zip = _zip_dir("model_comparison_results.zip", OUTPUT_DIR, exclude={"models"})
# 2) The trained models on their own (optional, large).
models_zip = _zip_dir("model_comparison_models.zip", OUTPUT_DIR, include={"models"})

for z in (results_zip, models_zip):
    print(f"{z.resolve()}  ({_human(z.stat().st_size)})")

# Try an in-browser download (Colab); otherwise print pull instructions (RunPod).
try:
    from google.colab import files  # type: ignore
    files.download(str(results_zip))
    # Uncomment to also pull the (large) models zip in Colab:
    # files.download(str(models_zip))
except Exception:
    print("\nNot on Colab — pull the archives off the pod, e.g.:")
    print(f"  runpodctl send {results_zip.resolve()}")
    print(f"  runpodctl send {models_zip.resolve()}")
    print("  # or:  scp root@<POD_IP>:<path> .   (Jupyter: right-click > Download)")